# CohortX Task 3 — genuine supervised training

This notebook does **not** contain a map from Test conditions to ICD codes.
It constructs `(condition, ICD family)` examples from the official Train
sheet and calls `LogisticRegression.fit` for KEEP, ASSOCIATION, and DIFF.
Validation is grouped by condition (nested leave-one-condition-out), never a
random pair split.  SapBERT is a frozen, offline feature encoder because five
unique training queries are far too few for defensible BERT fine-tuning.

Inputs:

- the official `Task_3.xlsx` and `mimic-iv_icd-10_dict.xlsx`;
- an offline copy of `cambridgeltl/SapBERT-from-PubMedBERT-fulltext`;
- optionally, its normalized ICD-title embedding cache and JSON sidecar.
  Without one, the notebook rebuilds the cache inside `/kaggle/working` from
  the attached official dictionary; it never publishes that derived file.

Outputs are written separately from the verified hand-built submissions.


In [1]:
from __future__ import annotations

import hashlib
import json
import os
import random
import re
import sys
import time
from collections import Counter
from dataclasses import asdict, dataclass
from itertools import product
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

# These must be set before importing Hugging Face libraries.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["NVIDIA_TF32_OVERRIDE"] = "0"

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


SEED = 20260716
MODEL_ID = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
MODEL_REVISION = "090663c3ae57bf35ffe4d0d468a2a88d03051a4d"
MODEL_SAFETENSORS_SHA256 = (
    "a4696930afef9aab296196d3d2142216c44cba24f21b4f285ceca7af21025614"
)
MODEL_CONFIG_SHA256 = (
    "c0fa35def52bf7d81865d23dba0417fdbb67d416fdde70cae5c7beab476b584e"
)
MODEL_VOCAB_SHA256 = (
    "79489a52be45e6fa033521e8ce8e4f62aedc0a742ee2aa6fc04667e5b0b1454d"
)
# Normalized-content fingerprints of the two official competition workbooks.
# These are opaque integrity guards, not labels or a condition-to-code map.
EXPECTED_ORDERED_CODE_SHA256 = (
    "0e8a31da47651dcc043e40f3fccda57c46d8cb9f36e4ced6607edf6d458036a2"
)
EXPECTED_ORDERED_TITLE_SHA256 = (
    "985a93e65020185cd5cd1b6065564e30084fee2ef89d2db5026e8dc2323894b0"
)
EXPECTED_TRAIN_ROWS_SHA256 = (
    "34274736254042c014412b5fb255a22dc60ef18cbfb0110d52d93a2a9b480949"
)
EXPECTED_TEST_ROWS_SHA256 = (
    "bd69347efcb5036254ba841e4c2e20336d14bc18d89ed70f82ba26bcac5e99e4"
)
POOLING_MODE = "attention_mask_mean"
MAX_SEQUENCE_LENGTH = 64
PADDING_MODE = "max_length"
ATTENTION_IMPLEMENTATION = "eager"
FIXED_INFERENCE_BATCH_SIZE = 64
TOKENIZER_MODE = "slow_wordpiece_lowercase"
TOKENIZER_LOWERCASE = True
BUCKETS = ("KEEP", "ASSOCIATION", "DIFF")
LABEL_BY_BUCKET = {"KEEP": 1, "ASSOCIATION": 2, "DIFF": 3}
EMPTY_TOKEN = "Not Applicable"
REQUIRED_COLUMNS = ("Condition", *BUCKETS)
TOP_HIT_CUTOFFS = (20, 40, 80, 120)
EXPECTED_NESTED_MODEL_F1 = 0.4176410161500671
EXPECTED_NESTED_RETRIEVAL_F1 = 0.45039275432189524
EXPECTED_NESTED_ENSEMBLE_F1 = 0.45039275432189524
EXPECTED_FINAL_RETRIEVAL_CONFIG = {
    "top_k": 80,
    "min_hits": 4,
    "min_mean": 0.60,
    "min_max": 0.60,
    "max_blocks": 4,
}

random.seed(SEED)
np.random.seed(SEED)


def _blocked_read_csv(*args, **kwargs):
    raise RuntimeError(
        "CSV input is disabled in this notebook. It must not read prior submissions."
    )


# The competition inputs are XLSX.  Blocking CSV reads makes accidental use of
# public-score artifacts or old submissions fail immediately.
pd.read_csv = _blocked_read_csv

ON_KAGGLE = Path("/kaggle/working").is_dir()
OUTPUT_DIR = Path(
    os.environ.get("COHORTX_OUTPUT_DIR", "/kaggle/working" if ON_KAGGLE else ".")
).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIT_EVENTS: list[dict[str, Any]] = []
AUDITED_INPUTS: list[str] = []

print(f"Python: {sys.version.split()[0]}")
print(f"numpy: {np.__version__}; pandas: {pd.__version__}; sklearn: {sklearn.__version__}")
print(f"Output directory: {OUTPUT_DIR}")


Python: 3.12.13
numpy: 2.0.2; pandas: 2.3.3; sklearn: 1.6.1
Output directory: /kaggle/working


## 1. Discover and validate only the approved inputs

The official workbooks are discovered together in one directory.  The
embedding cache is accepted only when its dynamic hashes match the current
sorted dictionary.  No workspace-specific dictionary hash is embedded here.


In [2]:
def norm_code(value: Any) -> str:
    if value is None:
        return ""
    try:
        if bool(value != value):
            return ""
    except (TypeError, ValueError):
        pass
    return re.sub(r"\s+", "", str(value)).upper().replace(".", "")


def parse_codes(value: Any, valid: set[str], *, strict: bool = True) -> set[str]:
    if value is None:
        return set()
    try:
        if bool(value != value):
            return set()
    except (TypeError, ValueError):
        pass
    text = str(value).strip()
    if not text or text.casefold() == EMPTY_TOKEN.casefold():
        return set()
    result = {norm_code(x) for x in re.split(r"[;,]", text) if norm_code(x)}
    invalid = result - valid
    if strict and invalid:
        raise ValueError(f"Invalid train codes: {sorted(invalid)[:10]}")
    return result & valid


def set_f1(pred: set[str], gold: set[str], *, empty_empty: float = 1.0) -> float:
    if not pred and not gold:
        return float(empty_empty)
    if not pred or not gold:
        return 0.0
    tp = len(pred & gold)
    return 2.0 * tp / (len(pred) + len(gold)) if tp else 0.0


def ordered_sha256(values: Sequence[str]) -> str:
    return hashlib.sha256("\n".join(values).encode("utf-8")).hexdigest()


def frame_sha256(frame: pd.DataFrame) -> str:
    rows = [
        "\t".join(str(value) for value in row)
        for row in frame.itertuples(index=False, name=None)
    ]
    return ordered_sha256(rows)


def file_sha256(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def _complete_data_root(path: Path) -> bool:
    return all(
        (path / name).is_file()
        for name in ("Task_3.xlsx", "mimic-iv_icd-10_dict.xlsx")
    )


def discover_data_root() -> Path:
    candidates: list[Path] = []
    if os.environ.get("COHORTX_DATA_DIR"):
        candidates.append(Path(os.environ["COHORTX_DATA_DIR"]).expanduser())
    candidates.append(Path.cwd())
    if Path("/kaggle/input").is_dir():
        candidates.extend(
            p.parent for p in Path("/kaggle/input").rglob("Task_3.xlsx")
        )
    matches = sorted(
        {p.resolve() for p in candidates if p.is_dir() and _complete_data_root(p)}
    )
    if len(matches) != 1:
        raise RuntimeError(
            "Expected exactly one directory containing both official workbooks; "
            f"found {matches}. Set COHORTX_DATA_DIR to disambiguate."
        )
    return matches[0]


@dataclass(frozen=True)
class CompetitionData:
    root: Path
    train: pd.DataFrame
    test: pd.DataFrame
    codes: tuple[str, ...]
    titles: tuple[str, ...]
    all_codes: frozenset[str]
    title_by_code: Mapping[str, str]
    gold: tuple[Mapping[str, set[str]], ...]


def load_competition_data() -> CompetitionData:
    root = discover_data_root()
    task_path = (root / "Task_3.xlsx").resolve()
    dict_path = (root / "mimic-iv_icd-10_dict.xlsx").resolve()
    AUDITED_INPUTS.extend([str(task_path), str(dict_path)])

    book = pd.ExcelFile(task_path)
    if not {"Train", "Test"}.issubset(book.sheet_names):
        raise ValueError(f"Required Train/Test sheets missing: {book.sheet_names}")
    train = pd.read_excel(task_path, sheet_name="Train", keep_default_na=False)
    test = pd.read_excel(task_path, sheet_name="Test", keep_default_na=False)
    for name, frame in (("Train", train), ("Test", test)):
        missing = [c for c in REQUIRED_COLUMNS if c not in frame.columns]
        if missing:
            raise ValueError(f"{name} is missing columns {missing}")
        if frame["Condition"].astype(str).str.strip().duplicated().any():
            raise ValueError(f"{name} contains duplicate conditions")
    train = train.loc[:, REQUIRED_COLUMNS].copy()
    test = test.loc[:, REQUIRED_COLUMNS].copy()
    train["Condition"] = train["Condition"].astype(str).str.strip()
    test["Condition"] = test["Condition"].astype(str).str.strip()

    icd = pd.read_excel(dict_path, keep_default_na=False)
    if not {"icd_code", "long_title"}.issubset(icd.columns):
        raise ValueError("ICD dictionary needs icd_code and long_title")
    icd = icd.loc[:, ["icd_code", "long_title"]].copy()
    icd["icd_code"] = icd["icd_code"].map(norm_code)
    icd["long_title"] = icd["long_title"].astype(str).str.strip()
    if (icd["icd_code"] == "").any() or icd["icd_code"].duplicated().any():
        raise ValueError("ICD dictionary has blank or duplicate normalized codes")
    icd = icd.sort_values("icd_code", kind="stable").reset_index(drop=True)
    codes = tuple(icd["icd_code"].tolist())
    titles = tuple(icd["long_title"].tolist())
    if any(len(code) < 3 for code in codes):
        raise ValueError("Every ICD code must have a 3-character family")
    all_codes = frozenset(codes)
    title_by_code = dict(zip(codes, titles))
    gold: list[Mapping[str, set[str]]] = []
    for row in train.itertuples(index=False):
        gold.append(
            {
                bucket: parse_codes(getattr(row, bucket), set(all_codes), strict=True)
                for bucket in BUCKETS
            }
        )
    return CompetitionData(
        root=root,
        train=train,
        test=test,
        codes=codes,
        titles=titles,
        all_codes=all_codes,
        title_by_code=title_by_code,
        gold=tuple(gold),
    )


data = load_competition_data()
expected_code_sha = ordered_sha256(data.codes)
expected_title_sha = ordered_sha256(data.titles)
train_rows_sha = frame_sha256(data.train)
test_rows_sha = frame_sha256(data.test)
input_fingerprints = {
    "ordered_code_sha256": (expected_code_sha, EXPECTED_ORDERED_CODE_SHA256),
    "ordered_title_sha256": (expected_title_sha, EXPECTED_ORDERED_TITLE_SHA256),
    "train_rows_sha256": (train_rows_sha, EXPECTED_TRAIN_ROWS_SHA256),
    "test_rows_sha256": (test_rows_sha, EXPECTED_TEST_ROWS_SHA256),
}
for name, (actual, expected) in input_fingerprints.items():
    if actual != expected:
        raise RuntimeError(
            f"Official-input content mismatch for {name}: {actual} != {expected}. "
            "Do not train on a different workbook revision."
        )
print(f"Train conditions: {len(data.train)}")
print(f"Test conditions:  {len(data.test)}")
print(f"ICD codes:        {len(data.codes):,}")
print(f"Official code/title hashes: {expected_code_sha[:12]} / {expected_title_sha[:12]}")
print(f"Official Train/Test hashes: {train_rows_sha[:12]} / {test_rows_sha[:12]}")


Train conditions: 5
Test conditions:  23
ICD codes:        97,441
Official code/title hashes: 0e8a31da4765 / 985a93e65020
Official Train/Test hashes: 342747362540 / bd69347efcb5


## 2. Load the offline encoder and aligned ICD-title cache

The cache is only a frozen input representation.  It is validated when
attached or rebuilt at runtime from the official dictionary.  The target
rankers below are fitted live from the competition Train labels.


In [3]:
def discover_embedding_cache() -> tuple[Path | None, Path | None]:
    candidates: list[Path] = []
    if os.environ.get("ICD_EMBEDDING_CACHE"):
        candidates.append(Path(os.environ["ICD_EMBEDDING_CACHE"]).expanduser())
    candidates.extend(Path.cwd().glob("cache/icd_emb__*SapBERT*.npy"))
    if Path("/kaggle/input").is_dir():
        candidates.extend(Path("/kaggle/input").rglob("icd_emb__*SapBERT*.npy"))
        candidates.extend(Path("/kaggle/input").rglob("icd_embeddings.npy"))
    valid: list[tuple[Path, Path]] = []
    for raw in candidates:
        path = raw.resolve()
        sidecars = [path.with_suffix(".json"), path.with_name("icd_embeddings_manifest.json")]
        meta_path = next((p for p in sidecars if p.is_file()), None)
        if path.is_file() and meta_path is not None:
            try:
                meta = json.loads(meta_path.read_text())
            except (OSError, json.JSONDecodeError):
                continue
            if meta.get("model_name", meta.get("model_id")) == MODEL_ID:
                required_geometry = {
                    "padding": PADDING_MODE,
                    "attention_implementation": ATTENTION_IMPLEMENTATION,
                    "runtime_batch_size": FIXED_INFERENCE_BATCH_SIZE,
                    "tokenizer": TOKENIZER_MODE,
                    "do_lower_case": TOKENIZER_LOWERCASE,
                }
                if any(meta.get(k) != v for k, v in required_geometry.items()):
                    print(f"Ignoring stale/non-v3 embedding cache: {path}")
                    continue
                valid.append((path, meta_path.resolve()))
    valid = sorted(set(valid))
    if len(valid) > 1:
        raise RuntimeError(
            "Found multiple SapBERT ICD embedding caches + JSON sidecars; "
            f"found {valid}. Set ICD_EMBEDDING_CACHE to disambiguate."
        )
    return valid[0] if valid else (None, None)


def discover_model_candidates() -> list[str]:
    candidates: list[Path] = []
    if os.environ.get("SAPBERT_DIR"):
        candidates.append(Path(os.environ["SAPBERT_DIR"]).expanduser())
    hf_root = (
        Path.home()
        / ".cache/huggingface/hub"
        / "models--cambridgeltl--SapBERT-from-PubMedBERT-fulltext/snapshots"
    )
    if hf_root.is_dir():
        candidates.extend(sorted(p for p in hf_root.iterdir() if p.is_dir()))
    search_roots = [Path.cwd()]
    if Path("/kaggle/input").is_dir():
        search_roots.append(Path("/kaggle/input"))
    for base in search_roots:
        for config in base.rglob("config.json"):
            parent = config.parent
            lower = str(parent).casefold()
            if "sapbert" in lower and any(
                (parent / name).exists()
                for name in ("tokenizer.json", "vocab.txt", "tokenizer_config.json")
            ):
                candidates.append(parent)
    return [str(p.resolve()) for p in dict.fromkeys(candidates) if p.is_dir()]


class DeterministicMeanEncoder:
    """Version-independent AutoModel encoder with explicit mean pooling."""

    def __init__(self, model_path: str, *, force_cpu: bool = False):
        import torch
        import transformers
        from transformers import AutoModel, AutoTokenizer

        path = Path(model_path).resolve()
        required_hashes = {
            "model.safetensors": MODEL_SAFETENSORS_SHA256,
            "config.json": MODEL_CONFIG_SHA256,
            "vocab.txt": MODEL_VOCAB_SHA256,
        }
        for filename, expected_hash in required_hashes.items():
            file_path = path / filename
            if not file_path.is_file():
                raise FileNotFoundError(f"Pinned SapBERT asset is missing {file_path}")
            actual_hash = file_sha256(file_path)
            if actual_hash != expected_hash:
                raise RuntimeError(
                    f"Pinned SapBERT hash mismatch for {filename}: "
                    f"{actual_hash} != {expected_hash}"
                )

        self.torch = torch
        self.device = torch.device(
            "cpu" if force_cpu or not torch.cuda.is_available() else "cuda"
        )
        torch.use_deterministic_algorithms(True)
        uses_ieee_precision_api = hasattr(torch.backends, "fp32_precision")
        if not uses_ieee_precision_api:
            torch.set_float32_matmul_precision("highest")
        if self.device.type == "cuda":
            if uses_ieee_precision_api:
                torch.backends.fp32_precision = "ieee"
                torch.backends.cuda.matmul.fp32_precision = "ieee"
                torch.backends.cudnn.fp32_precision = "ieee"
            else:
                torch.backends.cuda.matmul.allow_tf32 = False
                torch.backends.cudnn.allow_tf32 = False
            torch.backends.cudnn.benchmark = False
            torch.backends.cudnn.deterministic = True
            if hasattr(torch.backends.cuda, "enable_flash_sdp"):
                torch.backends.cuda.enable_flash_sdp(False)
            if hasattr(torch.backends.cuda, "enable_mem_efficient_sdp"):
                torch.backends.cuda.enable_mem_efficient_sdp(False)
            if hasattr(torch.backends.cuda, "enable_math_sdp"):
                torch.backends.cuda.enable_math_sdp(True)
        # Slow WordPiece tokenization plus explicit float32/safetensors removes
        # environment-dependent model/tokenizer selection on Kaggle.
        self.tokenizer = AutoTokenizer.from_pretrained(
            path,
            local_files_only=True,
            use_fast=False,
            do_lower_case=True,
        )
        if getattr(self.tokenizer, "do_lower_case", None) is not TOKENIZER_LOWERCASE:
            raise RuntimeError(
                "Tokenizer lowercase setting is not pinned to True: "
                f"{getattr(self.tokenizer, 'do_lower_case', None)!r}"
            )
        load_kwargs = {
            "local_files_only": True,
            "use_safetensors": True,
            "attn_implementation": ATTENTION_IMPLEMENTATION,
        }
        try:
            self.model = AutoModel.from_pretrained(
                path,
                dtype=torch.float32,
                **load_kwargs,
            )
        except TypeError:
            self.model = AutoModel.from_pretrained(
                path,
                torch_dtype=torch.float32,
                **load_kwargs,
            )
        self.model.to(self.device).eval()
        actual_attention = getattr(
            self.model.config, "_attn_implementation", ATTENTION_IMPLEMENTATION
        )
        if actual_attention != ATTENTION_IMPLEMENTATION:
            raise RuntimeError(
                f"Attention backend {actual_attention!r} != {ATTENTION_IMPLEMENTATION!r}"
            )
        self.dimension = int(self.model.config.hidden_size)
        self.model_path = path
        self.recommended_batch_size = FIXED_INFERENCE_BATCH_SIZE
        print(f"transformers: {transformers.__version__}; torch: {torch.__version__}")
        print(
            f"Encoder device={self.device}; pooling={POOLING_MODE}; "
            f"max_length={MAX_SEQUENCE_LENGTH}; padding={PADDING_MODE}; "
            f"attention={ATTENTION_IMPLEMENTATION}; "
            f"fixed_batch={FIXED_INFERENCE_BATCH_SIZE}; tokenizer={TOKENIZER_MODE}; "
            f"lowercase={TOKENIZER_LOWERCASE}"
        )

    def move_to_cpu(self) -> None:
        if self.device.type == "cpu":
            return
        self.model.to("cpu")
        self.device = self.torch.device("cpu")
        self.torch.cuda.empty_cache()
        print("CUDA preflight failed; encoder moved to deterministic CPU fallback")

    def encode(
        self,
        texts: Sequence[str],
        *,
        convert_to_numpy: bool = True,
        normalize_embeddings: bool = True,
        show_progress_bar: bool = False,
        batch_size: int = FIXED_INFERENCE_BATCH_SIZE,
    ) -> np.ndarray:
        del convert_to_numpy, show_progress_bar
        if batch_size != FIXED_INFERENCE_BATCH_SIZE:
            raise ValueError(
                f"batch_size={batch_size} must equal fixed deterministic batch "
                f"size {FIXED_INFERENCE_BATCH_SIZE}"
            )
        if not texts:
            return np.empty((0, self.dimension), dtype=np.float32)
        outputs: list[np.ndarray] = []
        with self.torch.inference_mode():
            for start in range(0, len(texts), batch_size):
                text_batch = list(texts[start : start + batch_size])
                real_rows = len(text_batch)
                if real_rows < batch_size:
                    text_batch.extend([text_batch[-1]] * (batch_size - real_rows))
                tokens = self.tokenizer(
                    text_batch,
                    padding=PADDING_MODE,
                    truncation=True,
                    max_length=MAX_SEQUENCE_LENGTH,
                    return_tensors="pt",
                )
                tokens = {key: value.to(self.device) for key, value in tokens.items()}
                hidden = self.model(**tokens).last_hidden_state
                mask = tokens["attention_mask"].unsqueeze(-1).to(hidden.dtype)
                pooled = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1e-9)
                if normalize_embeddings:
                    pooled = self.torch.nn.functional.normalize(pooled, dim=1)
                outputs.append(pooled[:real_rows].cpu().numpy().astype(np.float32))
        return np.concatenate(outputs, axis=0)


def load_offline_encoder():
    errors: list[str] = []
    for candidate in discover_model_candidates():
        try:
            model = DeterministicMeanEncoder(candidate)
            print(f"Loaded pinned deterministic encoder: {candidate}")
            return model, candidate
        except Exception as exc:
            errors.append(f"DeterministicMeanEncoder({candidate!r}): {exc}")
    raise RuntimeError("No offline SapBERT model could be loaded:\n" + "\n".join(errors))


encoder, encoder_source = load_offline_encoder()

# Fail before the 97k-title cache build if Kaggle changes pooling, tokenizer,
# weights, or precision semantics.  These values come from the pinned model
# with explicit float32 attention-mask mean pooling, not from any Test labels.
GOLDEN_PROBE_TEXTS = (
    "acute respiratory infection",
    "aortic aneurysm",
    "myocardial infarction",
)
GOLDEN_PROBE_FIRST16 = np.asarray(
    [
        [-0.01548940, -0.04649280, -0.02404778, -0.01110020, -0.01083193, 0.07536137, 0.00445637, -0.01328345, 0.01223498, 0.03301730, 0.00515232, 0.01700030, -0.02485915, 0.00535390, 0.00732113, -0.00503908],
        [-0.03025715, 0.00607710, -0.07122558, 0.06581908, -0.06609932, -0.03172617, 0.03886347, 0.02233210, 0.01597611, 0.01540115, -0.01101385, -0.01233010, 0.02050754, -0.00033408, -0.06252379, 0.01333561],
        [-0.08367646, 0.07547700, 0.01044879, -0.00103804, -0.00197490, 0.05502329, -0.00490987, 0.00606941, 0.01029280, -0.00719335, -0.00537095, 0.02375515, -0.02191839, 0.01162090, -0.02677447, 0.03499707],
    ],
    dtype=np.float32,
)
PREFLIGHT_TOP3_CODES = (
    ("J22", "J069", "J219"),
    ("I71", "Q2543", "I719"),
    ("I21", "I219", "I21A1"),
)
PREFLIGHT_TOP3_SCORES = np.asarray(
    [
        [0.86035502, 0.81425595, 0.73153150],
        [0.79615533, 0.79609311, 0.75522637],
        [0.87120426, 0.84846562, 0.79123962],
    ],
    dtype=np.float32,
)


def run_encoder_preflight() -> tuple[np.ndarray, float, float, float]:
    """Verify query, fixed-batch, and query/title geometry before the 97k build."""
    probe = encoder.encode(
        GOLDEN_PROBE_TEXTS,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    golden_error = float(
        np.max(np.abs(probe[:, :16] - GOLDEN_PROBE_FIRST16))
    )
    if golden_error > 2e-4:
        raise RuntimeError(
            "Pinned embedding golden probe failed before cache construction: "
            f"max_abs_error={golden_error:.8f}"
        )

    # V2 proved that a short-query golden probe alone was insufficient when
    # title batches used another padded shape. Encoder.encode now pads every
    # call to this exact fixed batch and fixed token length.
    large_texts = list(GOLDEN_PROBE_TEXTS)
    large_texts.extend(
        [GOLDEN_PROBE_TEXTS[-1]]
        * (encoder.recommended_batch_size - len(large_texts))
    )
    large_probe = encoder.encode(
        large_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=encoder.recommended_batch_size,
    )[: len(GOLDEN_PROBE_TEXTS)]
    shape_error = float(np.max(np.abs(large_probe - probe)))
    if shape_error > 2e-5:
        raise RuntimeError(
            "Small/large batch embedding invariance failed before cache "
            f"construction: max_abs_error={shape_error:.8f}"
        )

    preflight_codes = [code for group in PREFLIGHT_TOP3_CODES for code in group]
    title_texts = [data.title_by_code[code] for code in preflight_codes]
    title_vectors = encoder.encode(
        title_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=encoder.recommended_batch_size,
    )
    scores = np.asarray(
        [
            title_vectors[i * 3 : (i + 1) * 3] @ probe[i]
            for i in range(len(GOLDEN_PROBE_TEXTS))
        ],
        dtype=np.float32,
    )
    score_error = float(np.max(np.abs(scores - PREFLIGHT_TOP3_SCORES)))
    if score_error > 2e-3:
        raise RuntimeError(
            "Large-batch query/title semantic preflight failed before cache "
            f"construction: max_abs_error={score_error:.8f}; "
            f"scores={scores.tolist()}"
        )
    return probe, golden_error, shape_error, score_error


try:
    (
        golden_probe,
        golden_max_error,
        batch_shape_max_error,
        preflight_score_max_error,
    ) = run_encoder_preflight()
except RuntimeError as gpu_preflight_error:
    if encoder.device.type != "cuda":
        raise
    print(f"GPU preflight rejected: {gpu_preflight_error}")
    encoder.move_to_cpu()
    (
        golden_probe,
        golden_max_error,
        batch_shape_max_error,
        preflight_score_max_error,
    ) = run_encoder_preflight()

print(f"Pinned mean-pooling golden probe max error: {golden_max_error:.8f}")
print(
    "Fixed-shape semantic preflight: PASS "
    f"on {encoder.device} (vector_error={batch_shape_max_error:.8f}, "
    f"score_error={preflight_score_max_error:.8f})"
)


def build_runtime_embedding_cache() -> tuple[Path, Path]:
    """Build inside the writable run directory from the attached official data."""
    path = OUTPUT_DIR / "icd_embeddings_runtime.npy"
    meta_path = OUTPUT_DIR / "icd_embeddings_runtime.json"
    probe = encoder.encode(
        [data.titles[0]],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)
    dimension = int(probe.shape[1])
    matrix = np.lib.format.open_memmap(
        path,
        mode="w+",
        dtype=np.float32,
        shape=(len(data.titles), dimension),
    )
    batch_size = encoder.recommended_batch_size
    print(
        "No aligned input cache found; encoding ICD titles from the official "
        f"dictionary into {path} ..."
    )
    for start in range(0, len(data.titles), batch_size):
        end = min(start + batch_size, len(data.titles))
        title_batch = list(data.titles[start:end])
        if len(title_batch) < batch_size:
            title_batch.extend([title_batch[-1]] * (batch_size - len(title_batch)))
        matrix[start:end] = encoder.encode(
            title_batch,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=batch_size,
        )[: end - start].astype(np.float32)
        if start == 0 or end == len(data.titles) or end % 10_000 < batch_size:
            print(f"  encoded {end:,}/{len(data.titles):,}")
    matrix.flush()
    del matrix
    metadata = {
        "model_name": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "model_safetensors_sha256": MODEL_SAFETENSORS_SHA256,
        "pooling": POOLING_MODE,
        "max_sequence_length": MAX_SEQUENCE_LENGTH,
        "padding": PADDING_MODE,
        "attention_implementation": ATTENTION_IMPLEMENTATION,
        "runtime_batch_size": batch_size,
        "tokenizer": TOKENIZER_MODE,
        "do_lower_case": TOKENIZER_LOWERCASE,
        "dtype": "float32",
        "ordered_code_sha256": expected_code_sha,
        "ordered_title_sha256": expected_title_sha,
        "rows": len(data.codes),
        "dimension": dimension,
        "normalized": True,
        "built_from_official_dictionary_at_runtime": True,
        "file_sha256": file_sha256(path),
    }
    meta_path.write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\n")
    return path, meta_path


cache_path, cache_meta_path = discover_embedding_cache()
if cache_path is None or cache_meta_path is None:
    cache_path, cache_meta_path = build_runtime_embedding_cache()
    cache_origin = "rebuilt_from_attached_official_dictionary"
else:
    AUDITED_INPUTS.extend([str(cache_path), str(cache_meta_path)])
    cache_origin = "validated_input_cache"
cache_meta = json.loads(cache_meta_path.read_text())

if cache_meta.get("ordered_code_sha256") != expected_code_sha:
    raise RuntimeError("Embedding cache code-order hash does not match this dictionary")
if cache_meta.get("ordered_title_sha256") != expected_title_sha:
    raise RuntimeError("Embedding cache title-order hash does not match this dictionary")
if cache_meta.get("normalized") is not True:
    raise RuntimeError("Embedding cache must contain normalized vectors")
if cache_meta.get("pooling") != POOLING_MODE:
    raise RuntimeError(
        f"Embedding cache pooling {cache_meta.get('pooling')!r} != {POOLING_MODE!r}"
    )
if cache_meta.get("model_revision") != MODEL_REVISION:
    raise RuntimeError("Embedding cache was built from a different model revision")
if cache_meta.get("model_safetensors_sha256") != MODEL_SAFETENSORS_SHA256:
    raise RuntimeError("Embedding cache was built from different SapBERT weights")
if cache_meta.get("max_sequence_length") != MAX_SEQUENCE_LENGTH:
    raise RuntimeError("Embedding cache used a different maximum sequence length")
if cache_meta.get("tokenizer") != TOKENIZER_MODE:
    raise RuntimeError("Embedding cache used a different tokenizer backend")
if cache_meta.get("do_lower_case") is not TOKENIZER_LOWERCASE:
    raise RuntimeError("Embedding cache used a different lowercase setting")
if cache_meta.get("dtype") != "float32":
    raise RuntimeError("Embedding cache used a different numerical precision")
for cache_field, expected_value in (
    ("padding", PADDING_MODE),
    ("attention_implementation", ATTENTION_IMPLEMENTATION),
    ("runtime_batch_size", FIXED_INFERENCE_BATCH_SIZE),
):
    recorded_value = cache_meta.get(cache_field)
    if recorded_value != expected_value:
        raise RuntimeError(
            f"Embedding cache {cache_field}={recorded_value!r} != "
            f"{expected_value!r}"
        )
if cache_meta.get("file_sha256") and cache_meta["file_sha256"] != file_sha256(cache_path):
    raise RuntimeError("Embedding cache file SHA256 does not match its manifest")

code_embeddings = np.load(cache_path, mmap_mode="r")
expected_shape = (len(data.codes), int(cache_meta["dimension"]))
if code_embeddings.shape != expected_shape:
    raise RuntimeError(f"Embedding shape {code_embeddings.shape} != {expected_shape}")
probe_rows = np.linspace(0, len(data.codes) - 1, 9, dtype=int)
probe_norms = np.linalg.norm(np.asarray(code_embeddings[probe_rows]), axis=1)
if not np.allclose(probe_norms, 1.0, atol=2e-3):
    raise RuntimeError(f"Cache vectors are not normalized: {probe_norms}")

fresh_probe = encoder.encode(
    [data.titles[i] for i in probe_rows],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
).astype(np.float32)
alignment = np.sum(fresh_probe * np.asarray(code_embeddings[probe_rows]), axis=1)
if float(alignment.min()) < 0.995:
    raise RuntimeError(f"Model/cache alignment failed; probe cosines={alignment}")
print(f"Cache/model alignment min cosine: {alignment.min():.6f}")

# A query-only probe cannot detect a corrupt or differently encoded 97k-title
# matrix.  Verify the complete semantic retrieval path against three Train-only
# clinical phrases before fitting or predicting any Test row.
EXPECTED_SEMANTIC_TOP3 = (
    ("J22", "J069", "J219"),
    ("I71", "Q2543", "I719"),
    ("I21", "I219", "I21A1"),
)
EXPECTED_SEMANTIC_TOP1_SCORE = (0.86035502, 0.79615533, 0.87120426)
semantic_probe_rows: list[dict[str, Any]] = []
for query, query_vector, expected_top3, expected_top1_score in zip(
    GOLDEN_PROBE_TEXTS,
    golden_probe,
    EXPECTED_SEMANTIC_TOP3,
    EXPECTED_SEMANTIC_TOP1_SCORE,
):
    probe_scores = np.asarray(code_embeddings @ query_vector)
    top_rows = np.argsort(-probe_scores, kind="stable")[:3]
    actual_top3 = tuple(data.codes[int(i)] for i in top_rows)
    actual_top1_score = float(probe_scores[int(top_rows[0])])
    if actual_top3 != expected_top3 or abs(actual_top1_score - expected_top1_score) > 2e-3:
        raise RuntimeError(
            "Full semantic retrieval probe failed for "
            f"{query!r}: top3={actual_top3}, top1_score={actual_top1_score:.8f}; "
            f"expected {expected_top3}, {expected_top1_score:.8f}"
        )
    semantic_probe_rows.append(
        {
            "query": query,
            "top3": list(actual_top3),
            "top1_score": actual_top1_score,
        }
    )
print("Full-cache semantic probes:")
for row in semantic_probe_rows:
    print(f"  {row['query']}: {row['top3']} @ {row['top1_score']:.6f}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /kaggle/input/datasets/huynhnhuthuyk18hcm/sapbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


transformers: 5.0.0; torch: 2.10.0+cu128
Encoder device=cuda; pooling=attention_mask_mean; max_length=64; padding=max_length; attention=eager; fixed_batch=64; tokenizer=slow_wordpiece_lowercase; lowercase=True
Loaded pinned deterministic encoder: /kaggle/input/datasets/huynhnhuthuyk18hcm/sapbert
Pinned mean-pooling golden probe max error: 0.00000015
Fixed-shape semantic preflight: PASS on cuda (vector_error=0.00000000, score_error=0.00000077)
No aligned input cache found; encoding ICD titles from the official dictionary into /kaggle/working/icd_embeddings_runtime.npy ...
  encoded 64/97,441
  encoded 10,048/97,441
  encoded 20,032/97,441
  encoded 30,016/97,441
  encoded 40,000/97,441
  encoded 50,048/97,441
  encoded 60,032/97,441
  encoded 70,016/97,441
  encoded 80,000/97,441
  encoded 90,048/97,441
  encoded 97,441/97,441
Cache/model alignment min cosine: 1.000000
Full-cache semantic probes:
  acute respiratory infection: ['J22', 'J069', 'J219'] @ 0.860355
  aortic aneurysm: ['I71'

## 3. Build the ICD-family learning problem

Each example is one condition paired with one 3-character ICD family.  Labels
come solely from the Train sheet.  Family expansion matches the dominant
pattern in the provided labels and reduces 97,441 candidates to 1,914.


In [4]:
@dataclass(frozen=True)
class FamilyIndex:
    blocks: tuple[str, ...]
    starts: np.ndarray
    ends: np.ndarray
    sizes: np.ndarray
    representative_rows: np.ndarray
    centroids: np.ndarray
    neighbors_1: tuple[tuple[int, ...], ...]
    neighbors_2: tuple[tuple[int, ...], ...]


def build_family_index(
    codes: Sequence[str], titles: Sequence[str], embeddings: np.ndarray
) -> FamilyIndex:
    prefixes = np.asarray([code[:3] for code in codes], dtype=object)
    starts = np.r_[0, np.flatnonzero(prefixes[1:] != prefixes[:-1]) + 1]
    ends = np.r_[starts[1:], len(codes)]
    blocks = tuple(str(prefixes[i]) for i in starts)
    sizes = (ends - starts).astype(np.int32)
    representative_rows: list[int] = []
    centroids = np.empty((len(blocks), embeddings.shape[1]), dtype=np.float32)
    for j, (block, start, end) in enumerate(zip(blocks, starts, ends)):
        candidates = list(range(int(start), int(end)))
        exact = [i for i in candidates if codes[i] == block]
        representative_rows.append(
            exact[0]
            if exact
            else min(candidates, key=lambda i: (len(codes[i]), len(titles[i]), codes[i]))
        )
        centroid = np.asarray(embeddings[int(start) : int(end)], dtype=np.float32).mean(0)
        centroid /= max(float(np.linalg.norm(centroid)), 1e-12)
        centroids[j] = centroid

    lookup = {block: i for i, block in enumerate(blocks)}
    neighbors_1: list[tuple[int, ...]] = []
    neighbors_2: list[tuple[int, ...]] = []
    for block in blocks:
        match = re.fullmatch(r"([A-Z])(\d{2})", block)
        if not match:
            neighbors_1.append(())
            neighbors_2.append(())
            continue
        chapter, raw_number = match.groups()
        number = int(raw_number)
        n1 = tuple(
            lookup[key]
            for delta in (-1, 1)
            if 0 <= number + delta <= 99
            and (key := f"{chapter}{number + delta:02d}") in lookup
        )
        n2 = tuple(
            lookup[key]
            for delta in (-2, -1, 1, 2)
            if 0 <= number + delta <= 99
            and (key := f"{chapter}{number + delta:02d}") in lookup
        )
        neighbors_1.append(n1)
        neighbors_2.append(n2)
    return FamilyIndex(
        blocks=blocks,
        starts=starts.astype(np.int32),
        ends=ends.astype(np.int32),
        sizes=sizes,
        representative_rows=np.asarray(representative_rows, dtype=np.int32),
        centroids=centroids,
        neighbors_1=tuple(neighbors_1),
        neighbors_2=tuple(neighbors_2),
    )


families = build_family_index(data.codes, data.titles, code_embeddings)
block_to_index = {block: i for i, block in enumerate(families.blocks)}
family_code_sets = tuple(
    frozenset(data.codes[int(start) : int(end)])
    for start, end in zip(families.starts, families.ends)
)


def build_targets() -> np.ndarray:
    targets = np.zeros((len(data.train), len(families.blocks)), dtype=np.int8)
    for row_index, gold in enumerate(data.gold):
        for bucket, label in LABEL_BY_BUCKET.items():
            for code in gold[bucket]:
                family_index = block_to_index[code[:3]]
                old = int(targets[row_index, family_index])
                if old not in (0, label):
                    raise ValueError(
                        f"Family overlap in row {row_index}, {code[:3]}: {old} vs {label}"
                    )
                targets[row_index, family_index] = label
    return targets


targets = build_targets()
print(f"ICD families: {len(families.blocks):,}")
for bucket, label in LABEL_BY_BUCKET.items():
    print(f"Positive {bucket} condition-family pairs: {int((targets == label).sum())}")


ICD families: 1,914
Positive KEEP condition-family pairs: 64
Positive ASSOCIATION condition-family pairs: 6
Positive DIFF condition-family pairs: 16


## 4. Automatic features (no condition-keyed aliases)

Features combine frozen biomedical semantic similarity, word/character
TF-IDF, family size, retrieval density, and neighboring-family evidence.
Typos and acronyms are handled by generic dictionary-derived algorithms; no
Test condition appears as a key in a Python dictionary.


In [5]:
TOKEN_RE = re.compile(r"[A-Za-z]+")
QUERY_PART_RE = re.compile(r"[A-Za-z]+|[^A-Za-z]+")
PHRASE_STOP = {
    "of",
    "the",
    "and",
    "or",
    "with",
    "without",
    "to",
    "in",
    "on",
    "for",
    "a",
    "an",
    "due",
    "not",
    "elsewhere",
    "classified",
    "site",
    "sites",
    "multiple",
    "unspecified",
    "other",
    "specified",
    "following",
}
ADMIN_WORDS = {
    "right",
    "left",
    "bilateral",
    "initial",
    "subsequent",
    "sequela",
    "encounter",
}


def _aggregate(scores: np.ndarray, index: FamilyIndex) -> np.ndarray:
    output = np.empty((len(index.blocks), 4), dtype=np.float32)
    for j, (start, end) in enumerate(zip(index.starts, index.ends)):
        values = np.asarray(scores[int(start) : int(end)], dtype=np.float32)
        descending = np.sort(values)[::-1]
        output[j] = (
            descending[0],
            descending[: min(3, len(descending))].mean(),
            descending[: min(10, len(descending))].mean(),
            values.mean(),
        )
    return output


def _rank_and_hit_features(
    code_scores: np.ndarray, index: FamilyIndex, cutoffs: Sequence[int]
) -> tuple[np.ndarray, np.ndarray]:
    result = np.zeros((len(index.blocks), len(cutoffs)), dtype=np.float32)
    code_to_family = np.repeat(np.arange(len(index.blocks)), index.sizes)
    order = np.argsort(-code_scores, kind="stable")
    ranks = np.empty(len(code_scores), dtype=np.int32)
    ranks[order] = np.arange(len(code_scores), dtype=np.int32)
    min_family_rank = np.minimum.reduceat(ranks, index.starts)
    for column, cutoff in enumerate(cutoffs):
        counts = np.bincount(
            code_to_family[order[: min(cutoff, len(order))]],
            minlength=len(index.blocks),
        )
        result[:, column] = counts
    return -np.log1p(min_family_rank.astype(np.float32)), result


def _edit_distance_at_most_one(left: str, right: str) -> int | None:
    """Return exact distance 0/1, otherwise None, without a fuzzy library."""
    if abs(len(left) - len(right)) > 1:
        return None
    if left == right:
        return 0
    if len(left) == len(right):
        return 1 if sum(a != b for a, b in zip(left, right)) == 1 else None
    shorter, longer = (left, right) if len(left) < len(right) else (right, left)
    i = j = edits = 0
    while i < len(shorter) and j < len(longer):
        if shorter[i] == longer[j]:
            i += 1
            j += 1
        else:
            edits += 1
            j += 1
            if edits > 1:
                return None
    edits += int(j < len(longer))
    return edits if edits <= 1 else None


class FeatureBuilder:
    def __init__(
        self,
        competition_data: CompetitionData,
        family_index: FamilyIndex,
        embeddings: np.ndarray,
        text_encoder,
        known_conditions: Sequence[str],
    ):
        self.data = competition_data
        self.families = family_index
        self.embeddings = embeddings
        self.encoder = text_encoder
        print("Fitting unsupervised TF-IDF vocabularies on ICD titles ...")
        self.word_vectorizer = TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            max_features=100_000,
            sublinear_tf=True,
            dtype=np.float32,
        )
        self.char_vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            lowercase=True,
            ngram_range=(3, 5),
            min_df=2,
            max_features=80_000,
            sublinear_tf=True,
            dtype=np.float32,
        )
        self.word_titles = self.word_vectorizer.fit_transform(self.data.titles)
        self.char_titles = self.char_vectorizer.fit_transform(self.data.titles)
        self.vocabulary = Counter(
            token.casefold()
            for title in self.data.titles
            for token in TOKEN_RE.findall(title)
        )

        # Only initials requested by the current Train/Test strings (and their
        # one-letter deletions) need to be indexed.  The phrases themselves are
        # mined generically from ICD titles and never keyed by a condition.
        target_initials: set[str] = set()
        for condition in known_conditions:
            for token in TOKEN_RE.findall(str(condition)):
                if token.isupper() and 2 <= len(token) <= 8:
                    target_initials.add(token)
                    target_initials.update(
                        token[:j] + token[j + 1 :] for j in range(len(token))
                    )
        self.phrase_index: dict[str, Counter[tuple[str, ...]]] = {
            initials: Counter() for initials in target_initials
        }
        for title in self.data.titles:
            words = [word.casefold() for word in TOKEN_RE.findall(title)]
            clean = [word for word in words if word not in PHRASE_STOP]
            for ngram_size in range(2, 7):
                for start in range(len(clean) - ngram_size + 1):
                    phrase = tuple(clean[start : start + ngram_size])
                    if any(word in ADMIN_WORDS for word in phrase):
                        continue
                    initials = "".join(word[0] for word in phrase).upper()
                    if initials in self.phrase_index:
                        self.phrase_index[initials][phrase] += 1

    def correct_query(self, condition: str) -> str:
        output: list[str] = []
        for part in QUERY_PART_RE.findall(str(condition)):
            if not part.isalpha():
                output.append(part)
                continue
            lower = part.casefold()
            if (
                lower in self.vocabulary
                or (part.isupper() and 2 <= len(part) <= 8)
                or len(part) < 5
            ):
                output.append(part)
                continue
            candidates: list[tuple[int, int, str]] = []
            for word, frequency in self.vocabulary.items():
                if abs(len(word) - len(lower)) > 1:
                    continue
                distance = _edit_distance_at_most_one(lower, word)
                if distance is not None:
                    candidates.append((distance, -frequency, word))
            if candidates:
                replacement = min(candidates)[2]
                if part[0].isupper():
                    replacement = replacement.capitalize()
                part = replacement
            output.append(part)
        return "".join(output).strip()

    def expand_acronym(self, acronym: str, limit: int = 3) -> list[str]:
        exact = self.phrase_index.get(acronym, Counter())
        if exact:
            best_count = max(exact.values())
            tied = sorted(phrase for phrase, count in exact.items() if count == best_count)
            return [" ".join(phrase) for phrase in tied[:limit]]

        deletion_keys = [acronym[:j] + acronym[j + 1 :] for j in range(len(acronym))]
        partials: list[tuple[tuple[str, ...], int]] = []
        for key in deletion_keys:
            partials.extend(self.phrase_index.get(key, Counter()).items())
        synthesized: dict[tuple[str, ...], tuple[int, int, int]] = {}
        for a_index, (left, left_count) in enumerate(partials):
            for right, right_count in partials[a_index + 1 :]:
                common_length = 0
                for left_word, right_word in zip(left, right):
                    if left_word != right_word:
                        break
                    common_length += 1
                if common_length < 2:
                    continue
                common = left[:common_length]
                for first_tail, second_tail in (
                    (left[common_length:], right[common_length:]),
                    (right[common_length:], left[common_length:]),
                ):
                    phrase = common + first_tail + second_tail
                    initials = "".join(word[0] for word in phrase).upper()
                    if initials != acronym:
                        continue
                    score = (
                        common_length,
                        min(left_count, right_count),
                        left_count + right_count,
                    )
                    synthesized[phrase] = max(score, synthesized.get(phrase, (-1, -1, -1)))
        if not synthesized:
            return []
        best_score = max(synthesized.values())
        tied = sorted(phrase for phrase, score in synthesized.items() if score == best_score)
        return [" ".join(phrase) for phrase in tied[:limit]]

    def query_variants(self, condition: str) -> list[str]:
        raw = str(condition).strip()
        corrected = self.correct_query(raw)
        variants = [raw]
        if corrected.casefold() != raw.casefold():
            variants.append(corrected)
        for token in TOKEN_RE.findall(corrected):
            if token.isupper() and 2 <= len(token) <= 8:
                variants.extend(self.expand_acronym(token))
        return list(dict.fromkeys(value.strip() for value in variants if value.strip()))

    def _channel_features(self, scores: np.ndarray) -> np.ndarray:
        aggregate = _aggregate(scores, self.families)
        rank, hits = _rank_and_hit_features(scores, self.families, TOP_HIT_CUTOFFS)
        return np.column_stack([aggregate, rank, hits]).astype(np.float32)

    def transform_one(self, condition: str) -> tuple[np.ndarray, np.ndarray]:
        variants = self.query_variants(condition)
        query_embeddings = self.encoder.encode(
            variants,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32)
        semantic_scores = np.asarray(
            self.embeddings @ query_embeddings.T, dtype=np.float32
        ).max(axis=1)
        word_queries = self.word_vectorizer.transform(variants)
        char_queries = self.char_vectorizer.transform(variants)
        word_scores = np.asarray((self.word_titles @ word_queries.T).toarray()).max(axis=1)
        char_scores = np.asarray((self.char_titles @ char_queries.T).toarray()).max(axis=1)
        matrix = np.column_stack(
            [
                self._channel_features(semantic_scores),
                self._channel_features(word_scores),
                self._channel_features(char_scores),
                np.log1p(self.families.sizes.astype(np.float32)),
            ]
        ).astype(np.float32)
        if not np.isfinite(matrix).all():
            raise RuntimeError(f"Non-finite features for condition {condition!r}")
        return matrix, semantic_scores

    def transform(self, conditions: Sequence[str]) -> tuple[np.ndarray, list[np.ndarray]]:
        matrices: list[np.ndarray] = []
        semantic_scores: list[np.ndarray] = []
        for i, condition in enumerate(conditions, start=1):
            matrix, scores = self.transform_one(str(condition))
            matrices.append(matrix)
            semantic_scores.append(scores)
            print(
                f"  features {i:>2}/{len(conditions)}: {condition} "
                f"-> {self.query_variants(str(condition))}"
            )
        return np.stack(matrices), semantic_scores


train_conditions = data.train["Condition"].astype(str).tolist()
test_conditions = data.test["Condition"].astype(str).tolist()
feature_builder = FeatureBuilder(
    data,
    families,
    code_embeddings,
    encoder,
    known_conditions=[*train_conditions, *test_conditions],
)
print("Building Train features ...")
train_features, train_semantic_scores = feature_builder.transform(train_conditions)
print(f"Train feature tensor: {train_features.shape}")


Fitting unsupervised TF-IDF vocabularies on ICD titles ...
Building Train features ...
  features  1/5: URTI -> ['URTI', 'upper respiratory tract infection', 'upper respiratory tract infections', 'upper respiratory tract inflammation']
  features  2/5: Aortic Aneurysm -> ['Aortic Aneurysm']
  features  3/5: Ischemic Heart Disease -> ['Ischemic Heart Disease']
  features  4/5: Stroke -> ['Stroke']
  features  5/5: Shortness of Breadth -> ['Shortness of Breadth', 'Shortness of Breath']
Train feature tensor: (5, 1914, 28)


## 5. Nested leave-one-condition-out training

The split unit is the condition, so no pair from the held-out query leaks into
training.  Inner LOO chooses regularization, positive weight, and the number
of predicted families.  The outer fold is used only for honest evaluation.


In [6]:
@dataclass(frozen=True)
class ModelConfig:
    c: float
    positive_weight: float | str


@dataclass(frozen=True)
class DecoderConfig:
    top_n: int


MODEL_GRID = tuple(
    ModelConfig(c, positive_weight)
    for c in (0.001, 0.01, 0.1, 1.0)
    for positive_weight in (3.0, 10.0, 30.0, "balanced")
)
TOP_N_GRID = {
    "KEEP": (1, 2, 3, 4, 6, 8, 12, 16, 20, 24),
    "ASSOCIATION": (0, 1, 2, 3, 4),
    "DIFF": (0, 1, 2, 3, 4, 6),
}


def fit_ranker(
    features: np.ndarray,
    binary_targets: np.ndarray,
    condition_indices: Sequence[int],
    config: ModelConfig,
    *,
    bucket: str,
    stage: str,
) -> Pipeline:
    condition_indices = list(condition_indices)
    x = features[condition_indices].reshape(-1, features.shape[-1])
    y = binary_targets[condition_indices].reshape(-1)
    if len(np.unique(y)) != 2:
        raise RuntimeError(f"{bucket} training fold has only one class")
    class_weight: str | dict[int, float]
    if config.positive_weight == "balanced":
        class_weight = "balanced"
    else:
        class_weight = {0: 1.0, 1: float(config.positive_weight)}
    pipeline = Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "ranker",
                LogisticRegression(
                    C=config.c,
                    class_weight=class_weight,
                    solver="liblinear",
                    max_iter=1_500,
                    random_state=0,
                ),
            ),
        ]
    )
    pipeline.fit(x, y)
    FIT_EVENTS.append(
        {
            "bucket": bucket,
            "stage": stage,
            "conditions": [train_conditions[i] for i in condition_indices],
            "rows": int(len(x)),
            "positive_rows": int(y.sum()),
            "config": asdict(config),
        }
    )
    return pipeline


def predict_probability(model: Pipeline, matrix: np.ndarray) -> np.ndarray:
    return model.predict_proba(matrix)[:, 1].astype(np.float32)


def decode_top_n(probability: np.ndarray, top_n: int) -> set[str]:
    if top_n <= 0:
        return set()
    chosen = np.argsort(-probability, kind="stable")[:top_n]
    return set().union(*(family_code_sets[int(i)] for i in chosen))


def binary_targets_for(bucket: str) -> np.ndarray:
    return (targets == LABEL_BY_BUCKET[bucket]).astype(np.int8)


def mean_bucket_f1(
    probabilities: Mapping[int, np.ndarray], indices: Sequence[int], bucket: str, top_n: int
) -> float:
    return float(
        np.mean(
            [
                set_f1(decode_top_n(probabilities[i], top_n), data.gold[i][bucket])
                for i in indices
            ]
        )
    )


def choose_model_and_decoder(
    available_indices: Sequence[int], bucket: str, *, stage: str
) -> tuple[ModelConfig, DecoderConfig, float]:
    available = tuple(available_indices)
    if len(available) < 3:
        raise ValueError("At least three condition groups are required for inner LOO")
    y = binary_targets_for(bucket)
    best: tuple[float, int, float, float, ModelConfig, DecoderConfig] | None = None
    for config in MODEL_GRID:
        fold_probabilities: dict[int, np.ndarray] = {}
        for holdout in available:
            inner_train = [i for i in available if i != holdout]
            model = fit_ranker(
                train_features,
                y,
                inner_train,
                config,
                bucket=bucket,
                stage=f"{stage}/inner",
            )
            fold_probabilities[holdout] = predict_probability(
                model, train_features[holdout]
            )
        for top_n in TOP_N_GRID[bucket]:
            score = mean_bucket_f1(fold_probabilities, available, bucket, top_n)
            # Conservative deterministic tie-break: fewer families, stronger
            # regularization, then lower positive upweighting.
            weight_tiebreak = (
                -float(config.positive_weight)
                if config.positive_weight != "balanced"
                else -1_000_000.0
            )
            candidate = (
                score,
                -top_n,
                -config.c,
                weight_tiebreak,
                config,
                DecoderConfig(top_n),
            )
            if best is None or candidate[:4] > best[:4]:
                best = candidate
    assert best is not None
    return best[4], best[5], float(best[0])


def nested_group_oof() -> tuple[list[dict[str, set[str]]], list[dict[str, Any]]]:
    all_indices = tuple(range(len(train_conditions)))
    predictions: list[dict[str, set[str]]] = [dict() for _ in all_indices]
    audit: list[dict[str, Any]] = []
    for holdout in all_indices:
        outer_train = [i for i in all_indices if i != holdout]
        for bucket in BUCKETS:
            config, decoder, inner_score = choose_model_and_decoder(
                outer_train, bucket, stage=f"outer_{holdout}_{bucket}"
            )
            model = fit_ranker(
                train_features,
                binary_targets_for(bucket),
                outer_train,
                config,
                bucket=bucket,
                stage=f"outer_{holdout}/refit",
            )
            probability = predict_probability(model, train_features[holdout])
            predictions[holdout][bucket] = decode_top_n(probability, decoder.top_n)
            audit.append(
                {
                    "holdout": train_conditions[holdout],
                    "bucket": bucket,
                    "inner_f1": inner_score,
                    "model": asdict(config),
                    "decoder": asdict(decoder),
                }
            )
    return predictions, audit


started_cv = time.perf_counter()
nested_predictions, nested_audit = nested_group_oof()
oof_rows: list[dict[str, Any]] = []
for condition, pred, gold in zip(train_conditions, nested_predictions, data.gold):
    scores = {bucket: set_f1(pred[bucket], gold[bucket]) for bucket in BUCKETS}
    oof_rows.append(
        {
            "Condition": condition,
            **{f"F1_{bucket}": scores[bucket] for bucket in BUCKETS},
            "F1_avg": float(np.mean(list(scores.values()))),
            **{f"n_pred_{bucket}": len(pred[bucket]) for bucket in BUCKETS},
        }
    )
oof_frame = pd.DataFrame(oof_rows)
print(oof_frame.to_string(index=False))
observed_nested_model_f1 = float(oof_frame["F1_avg"].mean())
print(f"Nested grouped OOF macro-F1: {observed_nested_model_f1:.6f}")
if abs(observed_nested_model_f1 - EXPECTED_NESTED_MODEL_F1) > 2e-6:
    raise RuntimeError(
        "Train-only supervised regression checkpoint failed: "
        f"{observed_nested_model_f1:.12f} != {EXPECTED_NESTED_MODEL_F1:.12f}"
    )
print(f"Nested CV time: {time.perf_counter() - started_cv:.1f}s")


             Condition  F1_KEEP  F1_ASSOCIATION  F1_DIFF   F1_avg  n_pred_KEEP  n_pred_ASSOCIATION  n_pred_DIFF
                  URTI 0.046053             0.0      0.0 0.015351          152                   0            0
       Aortic Aneurysm 0.597015             0.0      0.0 0.199005           93                   0            0
Ischemic Heart Disease 0.711340             1.0      1.0 0.903780           81                   0            0
                Stroke 0.709202             0.0      1.0 0.569734          289                   0            0
  Shortness of Breadth 0.201005             1.0      0.0 0.400335           20                   0            0
Nested grouped OOF macro-F1: 0.417641
Nested CV time: 33.7s


### Automatic frozen-retrieval control and ensemble

A target model is fitted above, but with only five query groups a frozen
semantic retriever can generalize better.  Its configuration is selected in
the same outer/inner grouped protocol.  The practical output unions its KEEP
families with the supervised ranker; no code family is specified manually.


In [7]:
@dataclass(frozen=True)
class RetrievalConfig:
    top_k: int
    min_hits: int
    min_mean: float
    min_max: float
    max_blocks: int


RETRIEVAL_GRID = tuple(
    RetrievalConfig(*values)
    for values in product(
        (40, 80, 120),
        (2, 3, 4),
        (0.50, 0.55, 0.60),
        (0.60, 0.65),
        (4, 6, 8),
    )
)
CODE_TO_FAMILY = np.repeat(np.arange(len(families.blocks)), families.sizes)


def retrieval_keep(scores: np.ndarray, config: RetrievalConfig) -> set[str]:
    order = np.argsort(-scores, kind="stable")[: config.top_k]
    grouped: dict[int, list[float]] = {}
    for code_row in order:
        grouped.setdefault(int(CODE_TO_FAMILY[int(code_row)]), []).append(
            float(scores[int(code_row)])
        )
    accepted: list[tuple[int, int, float, float]] = []
    for family_index, values in grouped.items():
        hits = len(values)
        mean_score = float(np.mean(values))
        max_score = float(np.max(values))
        if hits < config.min_hits:
            continue
        if mean_score < config.min_mean and max_score < config.min_max:
            continue
        accepted.append((family_index, hits, mean_score, max_score))
    accepted.sort(
        key=lambda item: (
            -item[2],
            -item[3],
            -item[1],
            families.blocks[item[0]],
        )
    )
    chosen = [item[0] for item in accepted[: config.max_blocks]]
    return set().union(*(family_code_sets[i] for i in chosen)) if chosen else set()


def choose_retrieval_config(indices: Sequence[int]) -> tuple[RetrievalConfig, float]:
    best: tuple[float, tuple[Any, ...], RetrievalConfig] | None = None
    for config in RETRIEVAL_GRID:
        score = float(
            np.mean(
                [
                    set_f1(
                        retrieval_keep(train_semantic_scores[i], config),
                        data.gold[i]["KEEP"],
                    )
                    for i in indices
                ]
            )
        )
        conservative = (
            -config.max_blocks,
            -config.top_k,
            config.min_hits,
            config.min_mean,
            config.min_max,
        )
        candidate = (score, conservative, config)
        if best is None or candidate[:2] > best[:2]:
            best = candidate
    assert best is not None
    return best[2], best[0]


retrieval_nested_predictions: list[set[str]] = []
retrieval_nested_audit: list[dict[str, Any]] = []
for holdout in range(len(train_conditions)):
    outer_train = [i for i in range(len(train_conditions)) if i != holdout]
    config, inner_score = choose_retrieval_config(outer_train)
    prediction = retrieval_keep(train_semantic_scores[holdout], config)
    retrieval_nested_predictions.append(prediction)
    retrieval_nested_audit.append(
        {
            "holdout": train_conditions[holdout],
            "inner_f1": inner_score,
            "config": asdict(config),
        }
    )

retrieval_macro_rows: list[float] = []
ensemble_macro_rows: list[float] = []
for i, gold in enumerate(data.gold):
    retrieval_scores = [
        set_f1(retrieval_nested_predictions[i], gold["KEEP"]),
        set_f1(set(), gold["ASSOCIATION"]),
        set_f1(set(), gold["DIFF"]),
    ]
    retrieval_macro_rows.append(float(np.mean(retrieval_scores)))
    ensemble_pred = {
        bucket: set(nested_predictions[i][bucket]) for bucket in BUCKETS
    }
    ensemble_pred["KEEP"] |= retrieval_nested_predictions[i]
    ensemble_pred["ASSOCIATION"] -= ensemble_pred["KEEP"]
    ensemble_pred["DIFF"] -= ensemble_pred["KEEP"] | ensemble_pred["ASSOCIATION"]
    ensemble_macro_rows.append(
        float(
            np.mean(
                [set_f1(ensemble_pred[bucket], gold[bucket]) for bucket in BUCKETS]
            )
        )
    )

oof_frame["F1_avg_auto_retrieval"] = retrieval_macro_rows
oof_frame["F1_avg_trained_ensemble"] = ensemble_macro_rows
print("\nNested grouped controls:")
print(f"- trained model:    {oof_frame['F1_avg'].mean():.6f}")
print(f"- auto retrieval:   {np.mean(retrieval_macro_rows):.6f}")
print(f"- trained ensemble: {np.mean(ensemble_macro_rows):.6f}")
print("Retrieval configs chosen by outer folds:")
for row in retrieval_nested_audit:
    print(f"  {row['holdout']}: {row['config']}")

final_retrieval_config, final_retrieval_selection_f1 = choose_retrieval_config(
    tuple(range(len(train_conditions)))
)
print(
    f"Final automatic retrieval config: {final_retrieval_config}; "
    f"grouped-selection KEEP F1={final_retrieval_selection_f1:.6f}"
)
observed_nested_retrieval_f1 = float(np.mean(retrieval_macro_rows))
observed_nested_ensemble_f1 = float(np.mean(ensemble_macro_rows))
if (
    abs(observed_nested_retrieval_f1 - EXPECTED_NESTED_RETRIEVAL_F1) > 2e-6
    or abs(observed_nested_ensemble_f1 - EXPECTED_NESTED_ENSEMBLE_F1) > 2e-6
    or asdict(final_retrieval_config) != EXPECTED_FINAL_RETRIEVAL_CONFIG
):
    raise RuntimeError(
        "Train-only retrieval regression checkpoint failed: "
        f"retrieval={observed_nested_retrieval_f1:.12f}, "
        f"ensemble={observed_nested_ensemble_f1:.12f}, "
        f"config={asdict(final_retrieval_config)}"
    )
print("Deterministic Train-only regression checkpoint: PASS")



Nested grouped controls:
- trained model:    0.417641
- auto retrieval:   0.450393
- trained ensemble: 0.450393
Retrieval configs chosen by outer folds:
  URTI: {'top_k': 80, 'min_hits': 4, 'min_mean': 0.6, 'min_max': 0.6, 'max_blocks': 4}
  Aortic Aneurysm: {'top_k': 80, 'min_hits': 4, 'min_mean': 0.6, 'min_max': 0.6, 'max_blocks': 4}
  Ischemic Heart Disease: {'top_k': 80, 'min_hits': 4, 'min_mean': 0.6, 'min_max': 0.6, 'max_blocks': 4}
  Stroke: {'top_k': 80, 'min_hits': 4, 'min_mean': 0.6, 'min_max': 0.6, 'max_blocks': 4}
  Shortness of Breadth: {'top_k': 80, 'min_hits': 4, 'min_mean': 0.6, 'min_max': 0.6, 'max_blocks': 4}
Final automatic retrieval config: RetrievalConfig(top_k=80, min_hits=4, min_mean=0.6, min_max=0.6, max_blocks=4); grouped-selection KEEP F1=0.551178
Deterministic Train-only regression checkpoint: PASS


## 6. Refit all three target models on all Train rows

Final hyperparameters are selected by grouped LOO over all five Train
conditions.  This score is useful for model selection but the nested score
above is the less biased performance estimate.


In [8]:
@dataclass
class FittedBundle:
    models: dict[str, Pipeline]
    decoders: dict[str, DecoderConfig]
    configs: dict[str, ModelConfig]
    selection_scores: dict[str, float]


all_train_indices = tuple(range(len(train_conditions)))
final_models: dict[str, Pipeline] = {}
final_decoders: dict[str, DecoderConfig] = {}
final_configs: dict[str, ModelConfig] = {}
selection_scores: dict[str, float] = {}
for bucket in BUCKETS:
    config, decoder, score = choose_model_and_decoder(
        all_train_indices, bucket, stage=f"final_select_{bucket}"
    )
    final_models[bucket] = fit_ranker(
        train_features,
        binary_targets_for(bucket),
        all_train_indices,
        config,
        bucket=bucket,
        stage="final_fit",
    )
    final_decoders[bucket] = decoder
    final_configs[bucket] = config
    selection_scores[bucket] = score
    print(
        f"{bucket}: config={config}, top_n={decoder.top_n}, "
        f"grouped-selection F1={score:.6f}"
    )

bundle = FittedBundle(
    models=final_models,
    decoders=final_decoders,
    configs=final_configs,
    selection_scores=selection_scores,
)
assert {event["bucket"] for event in FIT_EVENTS if event["stage"] == "final_fit"} == set(
    BUCKETS
)


KEEP: config=ModelConfig(c=0.001, positive_weight=10.0), top_n=2, grouped-selection F1=0.575181
ASSOCIATION: config=ModelConfig(c=0.001, positive_weight=3.0), top_n=0, grouped-selection F1=0.400000
DIFF: config=ModelConfig(c=0.001, positive_weight=3.0), top_n=0, grouped-selection F1=0.400000


## 7. Predict Test through the fitted models and verify invariants

The same pure prediction path is run in original and shuffled Test order.  A
condition-keyed answer map would fail this permutation-equivariance check.


In [9]:
def predict_from_features(matrix: np.ndarray, fitted: FittedBundle) -> list[dict[str, set[str]]]:
    predictions: list[dict[str, set[str]]] = []
    for condition_matrix in matrix:
        row: dict[str, set[str]] = {}
        for bucket in BUCKETS:
            probability = predict_probability(fitted.models[bucket], condition_matrix)
            row[bucket] = decode_top_n(probability, fitted.decoders[bucket].top_n)
        # Deterministic bucket priority prevents duplicate codes across columns.
        row["ASSOCIATION"] -= row["KEEP"]
        row["DIFF"] -= row["KEEP"] | row["ASSOCIATION"]
        predictions.append(row)
    return predictions


def add_automatic_retrieval(
    model_predictions: Sequence[Mapping[str, set[str]]],
    semantic_scores: Sequence[np.ndarray],
    config: RetrievalConfig,
) -> list[dict[str, set[str]]]:
    output: list[dict[str, set[str]]] = []
    for model_row, scores in zip(model_predictions, semantic_scores):
        row = {bucket: set(model_row[bucket]) for bucket in BUCKETS}
        row["KEEP"] |= retrieval_keep(scores, config)
        row["ASSOCIATION"] -= row["KEEP"]
        row["DIFF"] -= row["KEEP"] | row["ASSOCIATION"]
        output.append(row)
    return output


def predict_conditions(
    conditions: Sequence[str], fitted: FittedBundle
) -> tuple[list[dict[str, set[str]]], list[dict[str, set[str]]], np.ndarray]:
    features, semantic_scores = feature_builder.transform([str(x) for x in conditions])
    model_predictions = predict_from_features(features, fitted)
    ensemble_predictions = add_automatic_retrieval(
        model_predictions, semantic_scores, final_retrieval_config
    )
    return model_predictions, ensemble_predictions, features


print("Building Test features and predicting ...")
test_features, test_semantic_scores = feature_builder.transform(test_conditions)
test_predictions = predict_from_features(test_features, bundle)
test_ensemble_predictions = add_automatic_retrieval(
    test_predictions, test_semantic_scores, final_retrieval_config
)

rng = np.random.default_rng(SEED)
permutation = rng.permutation(len(test_conditions))
shuffled_conditions = [test_conditions[int(i)] for i in permutation]
shuffled_predictions, shuffled_ensemble_predictions, _ = predict_conditions(
    shuffled_conditions, bundle
)
restored: list[dict[str, set[str]] | None] = [None] * len(test_conditions)
restored_ensemble: list[dict[str, set[str]] | None] = [None] * len(test_conditions)
for shuffled_position, original_position in enumerate(permutation):
    restored[int(original_position)] = shuffled_predictions[shuffled_position]
    restored_ensemble[int(original_position)] = shuffled_ensemble_predictions[
        shuffled_position
    ]
assert restored == test_predictions, "Predictions changed when Test row order changed"
assert (
    restored_ensemble == test_ensemble_predictions
), "Ensemble predictions changed when Test row order changed"
permutation_ok = True

# Counterfactual unseen-query smoke test: same model path, dictionary-valid output.
counterfactual_predictions, counterfactual_ensemble, _ = predict_conditions(
    ["Acute otitis media"], bundle
)
for bucket in BUCKETS:
    assert counterfactual_predictions[0][bucket] <= set(data.all_codes)
    assert counterfactual_ensemble[0][bucket] <= set(data.all_codes)


def format_codes(codes: set[str]) -> str:
    return "; ".join(sorted(codes)) if codes else EMPTY_TOKEN


submission = pd.DataFrame(
    {
        "Condition": test_conditions,
        **{
            bucket: [format_codes(pred[bucket]) for pred in test_predictions]
            for bucket in BUCKETS
        },
    }
)
ensemble_submission = pd.DataFrame(
    {
        "Condition": test_conditions,
        **{
            bucket: [
                format_codes(pred[bucket]) for pred in test_ensemble_predictions
            ]
            for bucket in BUCKETS
        },
    }
)


def validate_submission(frame: pd.DataFrame) -> None:
    if list(frame.columns) != list(REQUIRED_COLUMNS):
        raise ValueError(f"Bad columns: {frame.columns.tolist()}")
    if frame["Condition"].tolist() != test_conditions:
        raise ValueError("Submission row order differs from the official Test sheet")
    for row in frame.itertuples(index=False):
        parsed = {
            bucket: parse_codes(getattr(row, bucket), set(data.all_codes), strict=True)
            for bucket in BUCKETS
        }
        if parsed["KEEP"] & parsed["ASSOCIATION"]:
            raise ValueError(f"KEEP/ASSOCIATION overlap for {row.Condition}")
        if parsed["KEEP"] & parsed["DIFF"]:
            raise ValueError(f"KEEP/DIFF overlap for {row.Condition}")
        if parsed["ASSOCIATION"] & parsed["DIFF"]:
            raise ValueError(f"ASSOCIATION/DIFF overlap for {row.Condition}")


validate_submission(submission)
validate_submission(ensemble_submission)
submission_path = OUTPUT_DIR / "submission_trained_model_v3.csv"
ensemble_submission_path = OUTPUT_DIR / "submission_trained_ensemble_v3.csv"
oof_path = OUTPUT_DIR / "trained_real_oof_v3.csv"
model_path = OUTPUT_DIR / "trained_real_models_v3.joblib"
manifest_path = OUTPUT_DIR / "trained_real_run_manifest_v3.json"
submission.to_csv(submission_path, index=False)
ensemble_submission.to_csv(ensemble_submission_path, index=False)
oof_frame.to_csv(oof_path, index=False)
joblib.dump(
    {
        "models": bundle.models,
        "decoders": {k: asdict(v) for k, v in bundle.decoders.items()},
        "configs": {k: asdict(v) for k, v in bundle.configs.items()},
        "automatic_retrieval_config": asdict(final_retrieval_config),
        "feature_dimension": int(train_features.shape[-1]),
        "blocks": families.blocks,
    },
    model_path,
)

manifest = {
    "pipeline": "genuine_supervised_family_ranker_v3",
    "model_revision": MODEL_REVISION,
    "model_safetensors_sha256": MODEL_SAFETENSORS_SHA256,
    "pooling": POOLING_MODE,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "golden_probe_max_abs_error": golden_max_error,
    "batch_shape_max_abs_error": batch_shape_max_error,
    "preflight_semantic_score_max_abs_error": preflight_score_max_error,
    "padding": PADDING_MODE,
    "attention_implementation": ATTENTION_IMPLEMENTATION,
    "runtime_batch_size": FIXED_INFERENCE_BATCH_SIZE,
    "tokenizer": TOKENIZER_MODE,
    "do_lower_case": TOKENIZER_LOWERCASE,
    "semantic_probe_rows": semantic_probe_rows,
    "seed": SEED,
    "approved_inputs": AUDITED_INPUTS,
    "data_root": str(data.root),
    "model_id": MODEL_ID,
    "encoder_source": encoder_source,
    "cache_path": str(cache_path),
    "cache_metadata_path": str(cache_meta_path),
    "cache_origin": cache_origin,
    "ordered_code_sha256": expected_code_sha,
    "ordered_title_sha256": expected_title_sha,
    "train_rows_sha256": train_rows_sha,
    "test_rows_sha256": test_rows_sha,
    "train_conditions": len(train_conditions),
    "test_conditions": len(test_conditions),
    "families": len(families.blocks),
    "feature_dimension": int(train_features.shape[-1]),
    "fit_event_count": len(FIT_EVENTS),
    "final_fit_buckets": sorted(
        event["bucket"] for event in FIT_EVENTS if event["stage"] == "final_fit"
    ),
    "nested_oof_macro_f1": float(oof_frame["F1_avg"].mean()),
    "nested_oof_auto_retrieval_macro_f1": float(
        oof_frame["F1_avg_auto_retrieval"].mean()
    ),
    "nested_oof_trained_ensemble_macro_f1": float(
        oof_frame["F1_avg_trained_ensemble"].mean()
    ),
    "selection_scores": bundle.selection_scores,
    "configs": {k: asdict(v) for k, v in bundle.configs.items()},
    "decoders": {k: asdict(v) for k, v in bundle.decoders.items()},
    "automatic_retrieval_config": asdict(final_retrieval_config),
    "automatic_retrieval_selection_keep_f1": final_retrieval_selection_f1,
    "permutation_equivariance": permutation_ok,
    "submission_sha256": file_sha256(submission_path),
    "ensemble_submission_sha256": file_sha256(ensemble_submission_path),
    "model_artifact_sha256": file_sha256(model_path),
}
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")

print("\nSaved genuine-training artifacts:")
for path in (
    submission_path,
    ensemble_submission_path,
    oof_path,
    model_path,
    manifest_path,
):
    print(f"- {path}")
print(f"Live fit() events: {len(FIT_EVENTS)}")
print(f"Permutation-equivariance: {permutation_ok}")
print("No existing submission, fitted model, score config, or knowledge map was read.")


Building Test features and predicting ...
  features  1/23: Epistaxis -> ['Epistaxis']
  features  2/23: Intracranial Pressure -> ['Intracranial Pressure']
  features  3/23: Chronic Obstructive Pulmonary Disease -> ['Chronic Obstructive Pulmonary Disease']
  features  4/23: Enlarged Mediastinum -> ['Enlarged Mediastinum']
  features  5/23: Gout -> ['Gout']
  features  6/23: Latent Adrenal Insufficiency -> ['Latent Adrenal Insufficiency']
  features  7/23: Dermatomycosis -> ['Dermatomycosis']
  features  8/23: Pleurisy -> ['Pleurisy']
  features  9/23: Bronchitis -> ['Bronchitis']
  features 10/23: Thyroiditis -> ['Thyroiditis']
  features 11/23: Nasopharyngeal Carcinoma -> ['Nasopharyngeal Carcinoma']
  features 12/23: CKD -> ['CKD', 'chronic kidney disease']
  features 13/23: Hypothyroidism -> ['Hypothyroidism']
  features 14/23: Hematemesis -> ['Hematemesis']
  features 15/23: Heart Failure -> ['Heart Failure']
  features 16/23: Hypergonadism -> ['Hypergonadism']
  features 17/23: UT